### Gold - exemplos de indicadores

Não gera tabela adicional; demonstra que o modelo atende diretamente às
perguntas de negócio do case, sem necessidade de reprocessamento pelo
analista.

In [ ]:
%run "../00_setup/00_setup_ambiente"

#### receita, pedidos e ticket médio por região/mês

In [ ]:
%sql
SELECT
  dt.ano_mes,
  f.regiao,
  ROUND(SUM(f.receita_liquida_item), 2) AS receita_liquida,
  COUNT(DISTINCT f.order_id) AS qtd_pedidos,
  ROUND(SUM(f.receita_liquida_item) / COUNT(DISTINCT f.order_id), 2) AS ticket_medio
FROM case_dados.fact_pedido_item f
JOIN case_dados.dim_tempo dt ON f.order_date = dt.data
WHERE f.status_pedido != 'Cancelado'
GROUP BY dt.ano_mes, f.regiao
ORDER BY dt.ano_mes, receita_liquida DESC

#### taxa de cancelamento por canal

In [ ]:
%sql
SELECT
  canal_venda,
  COUNT(DISTINCT order_id) AS total_pedidos,
  COUNT(DISTINCT CASE WHEN pedido_cancelado THEN order_id END) AS pedidos_cancelados,
  ROUND(COUNT(DISTINCT CASE WHEN pedido_cancelado THEN order_id END) * 100.0 / COUNT(DISTINCT order_id), 1) AS taxa_cancelamento_pct
FROM case_dados.fact_pedido_item
GROUP BY canal_venda
ORDER BY taxa_cancelamento_pct DESC

#### atraso de entrega por região de destino

In [ ]:
%sql
SELECT
  destino_regiao,
  COUNT(*) AS total_entregas,
  SUM(CASE WHEN delivery_status = 'Atrasado' THEN 1 ELSE 0 END) AS entregas_atrasadas,
  ROUND(AVG(tempo_entrega_dias), 1) AS tempo_medio_entrega_dias
FROM case_dados.fact_entrega
GROUP BY destino_regiao
ORDER BY entregas_atrasadas DESC

#### receita por categoria

In [ ]:
%sql
SELECT
  p.categoria,
  ROUND(SUM(f.receita_liquida_item), 2) AS receita_liquida,
  SUM(f.quantity) AS qtd_total_vendida
FROM case_dados.fact_pedido_item f
LEFT JOIN case_dados.dim_produto p ON f.product_id = p.product_id
WHERE f.status_pedido != 'Cancelado'
GROUP BY p.categoria
ORDER BY receita_liquida DESC

#### ocorrências por tipo e severidade

In [ ]:
%sql
SELECT event_type, severity, COUNT(*) AS qtd_tickets
FROM case_dados.fact_ocorrencia
GROUP BY event_type, severity
ORDER BY qtd_tickets DESC

#### receita por trimestre

In [ ]:
%sql
SELECT
  dt.ano_trimestre,
  ROUND(SUM(f.receita_liquida_item), 2) AS receita_liquida
FROM case_dados.fact_pedido_item f
JOIN case_dados.dim_tempo dt ON f.order_date = dt.data
WHERE f.status_pedido != 'Cancelado'
GROUP BY dt.ano_trimestre
ORDER BY dt.ano_trimestre